# Study 833 — Deflated Sharpe Ratio 🎏

**Try enough strategies and the luckiest one *always* dazzles.**

Bailey & López de Prado (2014) proved the arithmetic: run `N` **independent** strategies on a
tape with **zero** true edge, and the *best* sample Sharpe is not zero — it grows with `N`. Here
the best of **1,000** empty strategies posts an annualised Sharpe of
**+1.25** with a *t* of **+2.80**… and is, with
certainty, nothing. The **Deflated Sharpe Ratio** shrinks it back to a coin flip.

*Numbers below are the frozen headline (`docs/results.md`, sim fingerprint `f7e4b81df8a2`,
as-of 2026-06-30); the live cells run the fast synthetic checks. Signal is NONE **by
construction** — the tape is a certified null.*


## 1. A gorgeous backtest, built from nothing

We generate 1,000 strategies whose *true* Sharpe is **exactly zero** — pure noise, no edge, by construction. Then we keep the one with the best backtest. Watch what the 'winner' looks like.

In [1]:
R = dict(mean_col_sharpe=0.026, obs_max_sharpe=1.251, winner_naive_t=2.8,
         exp_max_sharpe=1.456, winner_dsr=0.324)
print(f"mean strategy Sharpe (the truth): {R['mean_col_sharpe']:+.2f}  -> nothing real")
print(f"BEST of 1,000 Sharpe (annualised): {R['obs_max_sharpe']:+.2f}  -> looks amazing")
print(f"  its naive t-stat               : {R['winner_naive_t']:+.2f}  -> looks significant")
print(f"  expected MAX under pure luck   : {R['exp_max_sharpe']:+.2f}  -> the bar luck alone clears")
print(f"  Deflated Sharpe Ratio          : {R['winner_dsr']:.2f}   -> a coin flip (needs >=0.95 to matter)")

mean strategy Sharpe (the truth): +0.03  -> nothing real
BEST of 1,000 Sharpe (annualised): +1.25  -> looks amazing
  its naive t-stat               : +2.80  -> looks significant
  expected MAX under pure luck   : +1.46  -> the bar luck alone clears
  Deflated Sharpe Ratio          : 0.32   -> a coin flip (needs >=0.95 to matter)


## 2. Why the luck bar rises with every rule

The more strategies you try, the luckier the luckiest one is — that is not intuition, it is a formula (the *expected maximum Sharpe*). Let's watch it climb, live, on tiny null pools, and check the formula nails it.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
from deflated_sharpe import data, strategy as st
import numpy as np
for N in (10, 100, 1000):
    panel = data.null_panel(N, n_days=1260, ann_vol=0.15, seed=1)
    be = st.best_sharpe_experiment(panel)
    print(f'N={N:>5}: best empty-strategy Sharpe {be["obs_max_sharpe_ann"]:+.2f}  '
          f'(formula says ~{be["exp_max_sharpe_ann"]:+.2f})')
print('\n-> more trials, a luckier winner -- with ZERO real edge behind any of them')

N=   10: best empty-strategy Sharpe +0.68  (formula says ~+0.86)
N=  100: best empty-strategy Sharpe +0.98  (formula says ~+1.09)
N= 1000: best empty-strategy Sharpe +1.24  (formula says ~+1.41)

-> more trials, a luckier winner -- with ZERO real edge behind any of them


## 3. The catch that spares an honest idea

If deflation just punished big Sharpes, it would be useless. It punishes *searching*. An honestly-good **single** strategy (a true Sharpe of 1.0, no search) keeps a high Deflated Sharpe Ratio — live:

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
from deflated_sharpe import data, strategy as st
honest = data.honest_strategy(n_days=1260, true_ann_sharpe=1.0, seed=833)
d = st.deflated_sharpe_ratio(honest, n_trials=1)   # a single hypothesis, no search
print(f'honest single strategy: Sharpe {d["sharpe_ann"]:+.2f}, DSR {d["dsr"]:.3f}  '
      f'-> survives (the correction spares genuine skill)')
null_pool = data.null_panel(1000, 1260, 0.15, 833)
win = null_pool[:, int(np.nanargmax(st.panel_sr_per_period(null_pool)))]
dn = st.deflated_sharpe_ratio(win, n_trials=1000)
print(f'best of 1,000 empties : Sharpe {dn["sharpe_ann"]:+.2f}, DSR {dn["dsr"]:.3f}  '
      f'-> fails (consistent with luck)')

honest single strategy: Sharpe +1.06, DSR 0.991  -> survives (the correction spares genuine skill)
best of 1,000 empties : Sharpe +1.25, DSR 0.324  -> fails (consistent with luck)


## 4. The honest verdict

On a tape with **zero** real edge, the best of 1,000 strategies looks like a Sharpe-1.25 winner (naive *t* = +2.80) — and is provably nothing. Out of sample it collapses (**+1.77 → -0.28**) and bleeds on costs. The Deflated Sharpe Ratio (**0.32**, a coin flip) sees straight through it, while sparing the honest single strategy (DSR **0.96**). **Signal: None** (nothing real), **Tradability: Mirage**, and *does the trial count inflate the best Sharpe?* — **Confirmed**. A Sharpe without its trial count is not evidence.